In [ ]:
import serial
import time
import sys
from PIL import Image
import io
import numpy as np
import scipy.cluster # Para kmeans e vq
import cv2 as cv # Importa OpenCV

class ArduinoCommunicator:
    def __init__(self, port="/dev/ttyUSB0", baudrate=9600, timeout=2):
        self.port = port
        self.baudrate = baudrate
        self.timeout = timeout
        self.ser = None
        self.connected = False

    def connect(self):
        try:
            self.ser = serial.Serial(self.port, self.baudrate, timeout=self.timeout)
            time.sleep(2) # Aguarda a inicialização da comunicação serial
            self.connected = True
            print(f"[ArduinoCommunicator] Conectado à porta serial: {self.port}")
            # Aguarda o Arduino ficar pronto
            start_time = time.time()
            while time.time() - start_time < 5: # Espera por até 5 segundos
                response = self.read_line()
                if response == "READY":
                    print("[ArduinoCommunicator] Arduino reportou READY.")
                    return True
                time.sleep(0.1)
            print("[ArduinoCommunicator] Arduino não reportou READY dentro do tempo limite.")
            self.disconnect()
            return False
        except serial.SerialException as e:
            print(f"[ArduinoCommunicator] ERRO CRÍTICO: Não foi possível abrir a porta serial {self.port}: {e}")
            print("[ArduinoCommunicator] Certifique-se de que o dispositivo está conectado e que você tem permissão.")
            self.connected = False
            return False

    def disconnect(self):
        if self.ser and self.ser.is_open:
            self.ser.close()
            self.connected = False
            print("[ArduinoCommunicator] Conexão serial encerrada.")

    def send_command(self, cmd):
        if not self.connected:
            print(f"[ArduinoCommunicator] Erro: Não conectado ao Arduino. Comando '{cmd}' não enviado.")
            return False
        try:
            self.ser.write((cmd + "\n").encode("utf-8"))
            print(f"[ArduinoCommunicator] Enviado: {cmd}") # Debugging
            return True
        except serial.SerialException as e:
            print(f"[ArduinoCommunicator] Erro ao enviar comando '{cmd}': {e}")
            self.connected = False
            return False

    def read_line(self):
        if not self.connected:
            return None
        try:
            line = self.ser.readline().decode("utf-8", errors="ignore").strip()
            if line: print(f"[ArduinoCommunicator] Recebido: {line}") # Debugging
            return line
        except serial.SerialException as e:
            print(f"[ArduinoCommunicator] Erro ao ler da serial: {e}")
            self.connected = False
            return None

    def wait_for_response(self, expected_response, timeout=10):
        start_time = time.time()
        while time.time() - start_time < timeout:
            response = self.read_line()
            if response == expected_response:
                return True
            time.sleep(0.1)
        print(f"[ArduinoCommunicator] Tempo limite excedido esperando por '{expected_response}'.")
        return False


class CubeScanner:
    def __init__(self, arduino_port="/dev/ttyUSB0", droidcam_url="http://YOUR_DROIDCAM_IP_ADDRESS:PORT/video"): # Adicionado droidcam_url
        self.comm = ArduinoCommunicator(port=arduino_port)
        self.faces_data = {}
        self.camera = None
        self.droidcam_url = droidcam_url
        self._setup_camera()

    def _setup_camera(self):
        try:
            # Inicializa a captura de vídeo da DroidCam
            self.camera = cv.VideoCapture(self.droidcam_url)
            if not self.camera.isOpened():
                raise IOError(f"Não foi possível abrir o fluxo de vídeo DroidCam em {self.droidcam_url}. Verifique o URL e a conexão.")
            print(f"[CubeScanner] DroidCam configurada com sucesso para URL: {self.droidcam_url}")
        except Exception as e:
            print(f"[CubeScanner] ERRO ao configurar a DroidCam: {e}.")
            self.camera = None

    def _capture_and_analyse_face(self):
        if not self.camera or not self.camera.isOpened():
            raise RuntimeError("Câmera DroidCam não está configurada ou disponível.")

        ret, frame = self.camera.read()
        if not ret:
            raise IOError("Não foi possível ler o frame da câmera DroidCam.")

        frame = cv.flip(frame, 1) # Inverte horizontalmente como no outro script

        # Converte o frame do OpenCV (array numpy) para uma imagem PIL
        img = Image.fromarray(cv.cvtColor(frame, cv.COLOR_BGR2RGB))

        # Processamento de imagem (restante do código original)
        img = img.rotate(46 + 270)
        img = img.crop((310, 445, 310 + 365, 445 + 365))

        width, _ = img.size
        incr = float(width) / 3.0

        colours = []
        for i in range(3):
            for j in range(3):
                # Encontra a cor dominante
                im = img.crop((incr * j, incr * i, incr * j + incr, incr * i + incr)).resize((50, 50))
                ar = np.asarray(im)
                shape = ar.shape
                ar = ar.reshape(np.prod(shape[:2]), shape[2]).astype(float)

                # Encontra clusters
                NUM_CLUSTERS = 6
                codes, dist = scipy.cluster.vq.kmeans(ar, NUM_CLUSTERS)

                vecs, dist = scipy.cluster.vq.vq(ar, codes)
                counts, bins = np.histogram(vecs, len(codes))

                index_max = np.argmax(counts)
                peak = codes[index_max]
                colours.append([peak[0], peak[1], peak[2]])

        # Converte para HSL (somente H) - Garante independência da iluminação
        hsl = []
        actual = []
        for colour in colours:
            r = colour[0] / 255.0
            g = colour[1] / 255.0
            b = colour[2] / 255.0
            mx = max(r, g, b)
            mn = min(r, g, b)

            l = (mn + mx) / 2.0

            if l < 0.5:
                s = (mx - mn) / (mx + mn)
            else:
                s = (mx - mn) / (2.0 - mx - mn)

            if mn == mx:
                s = 0

            if mx == r:
                h = (g - b) / (mx - mn)
            elif mx == g:
                h = 2.0 + (b - r) / (mx - mn)
            elif mx == b:
                h = 4.0 + (r - g) / (mx - mn)
            h *= 60
            if h < 0:
                h += 360
            hsl.append((h, s, l))
            if s < 0.45:
                actual.append("w")
            else:
                actual.append("unknown")
        colours = hsl

        known_colours = {"r": (342, 0.80, 0.5), "b": (194, 0.98, 0.5), "g": (128, 1.0, 0.5), "y": (78, 1.0, 0.5), "o": (13.7, 0.72, 0.5)}

        for i in range(len(colours)):
            if actual[i] != "w":
                best_dist = 99e9
                best_colour = "unknown"
                for key, value in known_colours.items():
                    h, s, l = value
                    s *= 360 # Normaliza
                    a, b, c = colours[i]
                    b *= 360 # Normaliza
                    dist = abs(h - a) + abs(s - b)
                    if dist < best_dist:
                        best_dist = dist
                        best_colour = key
                actual[i] = (best_colour)

        actual = ''.join(actual)
        print("Detected: {0}".format(actual))
        return actual

    def scan_cube_faces(self):
        if not self.comm.connect():
            print("Não foi possível conectar ao Arduino. Abortando varredura do cubo.")
            return None
        if not self.camera or not self.camera.isOpened(): # Verifica também se a DroidCam está aberta
            print("Câmera DroidCam não está disponível. Abortando varredura do cubo.")
            self.comm.disconnect()
            return None

        print("Iniciando varredura do cubo...")
        self.comm.send_command("RESET")
        if not self.comm.wait_for_response("READY"):
            self.comm.disconnect()
            return None

        scan_moves = [
            ("TOP", None), # Posição inicial, sem movimento
            ("FRONT", "MOVE U"), # Gira a base para a frente
            ("RIGHT", "MOVE U"), # Gira a base para a direita
            ("BACK", "MOVE U"),  # Gira a base para trás
            ("LEFT", "MOVE U"),  # Gira a base para a esquerda
            ("BOTTOM", "MOVE F") # Vira o cubo para mostrar a face inferior
        ]

        for face_name, arduino_cmd in scan_moves:
            print(f"\nPreparando para escanear a face: {face_name}")
            if arduino_cmd:
                self.comm.send_command(arduino_cmd)
                if not self.comm.wait_for_response("DONE"):
                    print(f"Erro ao mover o cubo para a face {face_name}. Abortando.")
                    self.comm.disconnect()
                    return None

            print(f"Capturando imagem e analisando a face: {face_name}...")
            try:
                detected_colors = self._capture_and_analyse_face()
                self.faces_data[face_name] = detected_colors
                print(f"Cores detectadas para {face_name}: {detected_colors}")
            except Exception as e:
                print(f"Erro ao analisar a imagem da face {face_name}: {e}")
                self.comm.disconnect()
                return None

        self.comm.send_command("FINISH") # Avisa o Arduino que a varredura terminou
        self.comm.wait_for_response("DONE") # Espera a confirmação de FINISH
        self.comm.disconnect()
        print("\nVarredura do cubo concluída.")
        return self.faces_data

# Exemplo de uso (para testes)
if __name__ == "__main__":
    print("Este módulo é projetado para ser importado por outros scripts, como o Resolvedor.")
    print("Iniciando um teste básico de varredura do cubo...")

    # Ajuste a porta serial conforme necessário (ex: 'COM3' no Windows, '/dev/ttyACM0' ou '/dev/ttyUSB0' no Linux)
    # Substitua "YOUR_DROIDCAM_IP_ADDRESS:PORT/video" pelo URL real do seu DroidCam.
    droidcam_url = "http://YOUR_DROIDCAM_IP_ADDRESS:PORT/video" # Exemplo de URL
    scanner = CubeScanner(arduino_port="/dev/ttyUSB0", droidcam_url=droidcam_url) # Substitua pela porta correta e URL
    cube_colors = scanner.scan_cube_faces()

    if cube_colors:
        print("\n--- Resultado Final da Varredura do Cubo ---")
        for face, colors in cube_colors.items():
            print(f"Face {face}: {colors}")
    else:
        print("A varredura do cubo falhou ou não produziu resultados.")